# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by @id
print("Record Sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}  name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}  name: {getattr(field, 'name', '<no_name>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's get all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record set @ids: {record_set_ids}")
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist() if not df.empty else 'No records'}")
# Display the first record set DataFrame's head, if available
if record_set_ids:
    example_rs = record_set_ids[0]
    if not dataframes[example_rs].empty:
        display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select an example record set for EDA
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[rs_id] if rs_id else None
if df is not None and not df.empty:
    # Guess likely numeric fields
    numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if not numeric_fields:
        # Try to cast any available columns to float
        possible_numeric = [c for c in df.columns if any(k in c.lower() for k in ['abundance', 'weight', 'count', 'coverage', 'mw', 'pi'])]
        for col in possible_numeric:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        # Show histogram
        print(f"Numeric field chosen: {numeric_field}")
        # Filter records above a threshold (10, for demonstration)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Grouping by a field
        group_fields = [c for c in df.columns if c != numeric_field]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field, dropna=True)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the first record set.")
else:
    print("No records found in any record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_fields:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # Relationship with group_field
    if group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Not enough numeric or categorical data to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded FAIR² Croissant dataset describing protein abundance and characteristics in extracellular vesicles from human mast cells.
- Identified available record sets and field `@id`s for targeted exploration.
- Demonstrated EDA techniques such as filtering, normalization, and grouping on a numeric field.
- Visualized main distributions and relationships in the dataset.

Next steps can include deeper domain-specific queries, additional visualizations, and preparation for downstream modeling tasks.